In [11]:
import numpy as np


# A simple 4-room grid-world implementation with a grid of 7x7 for a total of 20 states (the walls do not count!).
# We arbitrarily chose the actions '0' = 'go up', '1' = 'go right', '2'  = 'go down' thus '3' = 'go left'
# Finally the state '0' is the top-left corner, 'nS - 1' is the down-right corner.
# The agent is teleported back to the initial state '0' (top-left corner) ,  whenever performing any action in rewarding state '19' (down-right corner).



#modified with added toggle to teleport to any other S − 1 states with equal probability.
class Four_Room_Teleportation():

	def __init__(self,tele):
		self.nS = 20
		nS = self.nS
		self.nA = 4
		self.tele = tele

		self.map = [[-1, -1, -1, -1, -1, -1, -1],
					[-1,  0,  1,  2,  3,  4, -1],
					[-1,  5,  6, -1,  7,  8, -1],
					[-1,  9, -1, -1, 10, -1, -1],
					[-1, 11, 12, 13, 14, 15, -1],
					[-1, 16, 17, -1, 18, 19, -1],
					[-1, -1, -1, -1, -1, -1, -1]]
		map = np.array(self.map)

		# We build the transitions matrix P using the map.
		self.P = np.zeros((nS, 4, nS))
		for s in range(nS):
			temp = np.where(s == map)
			y, x = temp[0][0], temp[1][0]
			up = map[x, y-1]
			right = map[x+1, y]
			down = map[x, y+1]
			left = map[x-1, y]

			# Action 0: go up.
			a = 0
			self.P[s, a, s] += 0.1
			# Up
			if up == -1:
				self.P[s, a, s] += 0.7
			else:
				self.P[s, a, up] += 0.7
			# Right
			if right == -1:
				self.P[s, a, s] += 0.1
			else:
				self.P[s, a, right] += 0.1
			# Left
			if left == -1:
				self.P[s, a, s] += 0.1
			else:
				self.P[s, a, left] += 0.1
			
			# Action 1: go right.
			a = 1
			self.P[s, a, s] += 0.1
			# Up
			if up == -1:
				self.P[s, a, s] += 0.1
			else:
				self.P[s, a, up] += 0.1
			# Right
			if right == -1:
				self.P[s, a, s] += 0.7
			else:
				self.P[s, a, right] += 0.7
			# Down
			if down == -1:
				self.P[s, a, s] += 0.1
			else:
				self.P[s, a, down] += 0.1
			
			# Action 2: go down.
			a = 2
			self.P[s, a, s] += 0.1
			# Right
			if right == -1:
				self.P[s, a, s] += 0.1
			else:
				self.P[s, a, right] += 0.1
			# Down
			if down == -1:
				self.P[s, a, s] += 0.7
			else:
				self.P[s, a, down] += 0.7
			# Left
			if left == -1:
				self.P[s, a, s] += 0.1
			else:
				self.P[s, a, left] += 0.1

			# Action 3: go left.
			a = 3
			self.P[s, a, s] += 0.1
			# Up
			if up == -1:
				self.P[s, a, s] += 0.1
			else:
				self.P[s, a, up] += 0.1
			# Down
			if down == -1:
				self.P[s, a, s] += 0.1
			else:
				self.P[s, a, down] += 0.1
			# Left
			if left == -1:
				self.P[s, a, s] += 0.7
			else:
				self.P[s, a, left] += 0.7
			
			# Set to teleport back when in the rewarding state.
			if s == self.nS - 1:
				for a in range(4):
					for ss in range(self.nS):
						self.P[s, a, ss] = 0
						if self.tele and not(ss==self.nS-1):
							self.P[s, a, ss] = 1/(self.nS-1)
						if ss== 0 and not(self.tele):
							self.P[s, a, ss] = 1

			
		# We build the reward matrix R.
		self.R = np.zeros((nS, 4))
		for a in range(4):
			self.R[nS - 1, a] = 1

		# We (arbitrarily) set the initial state in the top-left corner.
		self.s = 0

	# To reset the environment in initial settings.
	def reset(self):
		self.s = 0
		return self.s

	# Perform a step in the environment for a given action. Return a couple state, reward (s_t, r_t).
	def step(self, action):
		new_s = np.random.choice(np.arange(self.nS), p=self.P[self.s, action])
		reward = self.R[self.s, action]
		self.s = new_s
		return new_s, reward










# A naive function to output a readable matrix from a policy on the 4-room environment.
def display_4room_policy(policy):
	map = np.array([[-1, -1, -1, -1, -1, -1, -1],
					[-1,  0,  1,  2,  3,  4, -1],
					[-1,  5,  6, -1,  7,  8, -1],
					[-1,  9, -1, -1, 10, -1, -1],
					[-1, 11, 12, 13, 14, 15, -1],
					[-1, 16, 17, -1, 18, 19, -1],
					[-1, -1, -1, -1, -1, -1, -1]])
	
	res = []

	for i in range(7):
		temp = []
		for j in range(7):
			if map[i][j] == -1:
				temp.append("Wall ")
			elif policy[map[i][j]] == 0:
				temp.append(" Up  ")
			elif policy[map[i][j]] == 1:
				temp.append("Right")
			elif policy[map[i][j]] == 2:
				temp.append("Down ")
			elif policy[map[i][j]] == 3:
				temp.append("Left ")
		
		res.append(temp)

	return np.array(res)

	


In [12]:
def VI_avg_reward(env, max_iter = 10**6, epsilon = 10**(-6)):

	# The variable containing the optimal policy estimate at the current iteration.
	policy = np.zeros(env.nS, dtype=int)
	niter = -1

	# Initialise the value and epsilon as proposed in the course.
	V1 = np.empty(env.nS) # initialize empty
	V0 = np.ones(env.nS)
	# The main loop of the Value Iteration algorithm.
	while True:
		niter += 1
		for s in range(env.nS):
			for a in range(env.nA):
				temp = env.R[s, a] + sum([V * p for (V, p) in zip(V0, env.P[s, a])]) # note gamma removed
				if (a == 0) or (temp > V1[s]):
					V1[s] = temp
					policy[s] = a
		

		# Testing the stopping criterion (+1 abitrary stop when 'max_iter' is reached).

		if((np.max(V1 - V0) - np.min(V1 - V0)) < epsilon):
			gain_estimate = 1/2*(np.max(V1 - V0) + np.min(V1 - V0))
			bias_span = np.max(V0) - np.min(V0)
			return niter, bias_span, policy, gain_estimate, V0
		
		else:
			V0 = V1
			V1 = np.empty(env.nS) # reinitialize to empty array
		if niter > max_iter:
			print("No convergence in VI after: ", max_iter, " steps!")
			return niter, V0, policy

In [20]:
# Tele indicates whether to teleport to the first state
env = Four_Room_Teleportation(tele = False)
n_iter, bias_span, optimal_pol, gain, value = VI_avg_reward(env)

print(bias_span)
print(gain)
print(optimal_pol)

0.9243739092688656
0.07562562057751432
[1 1 1 2 2 2 0 2 3 2 2 1 1 1 1 2 1 0 1 0]


In [14]:
display_4room_policy(optimal_pol)

array([['Wall ', 'Wall ', 'Wall ', 'Wall ', 'Wall ', 'Wall ', 'Wall '],
       ['Wall ', 'Right', 'Right', 'Right', 'Down ', 'Down ', 'Wall '],
       ['Wall ', 'Down ', ' Up  ', 'Wall ', 'Down ', 'Left ', 'Wall '],
       ['Wall ', 'Down ', 'Wall ', 'Wall ', 'Down ', 'Wall ', 'Wall '],
       ['Wall ', 'Right', 'Right', 'Right', 'Right', 'Down ', 'Wall '],
       ['Wall ', 'Right', ' Up  ', 'Wall ', 'Right', ' Up  ', 'Wall '],
       ['Wall ', 'Wall ', 'Wall ', 'Wall ', 'Wall ', 'Wall ', 'Wall ']],
      dtype='<U5')

In [19]:
env_tele = Four_Room_Teleportation(tele=True)
n_iter_tele, bias_span_tele, optimal_pol_tele, gain_tele,value_tele = VI_avg_reward(
    env_tele)
print(bias_span_tele)
print(gain_tele)
print(optimal_pol)

1.4408475030162995
0.11787970699374473
[1 1 1 2 2 2 0 2 3 2 2 1 1 1 1 2 1 0 1 0]


In [18]:
display_4room_policy(optimal_pol_tele)

array([['Wall ', 'Wall ', 'Wall ', 'Wall ', 'Wall ', 'Wall ', 'Wall '],
       ['Wall ', 'Right', 'Right', 'Right', 'Down ', 'Down ', 'Wall '],
       ['Wall ', 'Down ', ' Up  ', 'Wall ', 'Down ', 'Left ', 'Wall '],
       ['Wall ', 'Down ', 'Wall ', 'Wall ', 'Down ', 'Wall ', 'Wall '],
       ['Wall ', 'Right', 'Right', 'Right', 'Right', 'Down ', 'Wall '],
       ['Wall ', 'Right', ' Up  ', 'Wall ', 'Right', ' Up  ', 'Wall '],
       ['Wall ', 'Wall ', 'Wall ', 'Wall ', 'Wall ', 'Wall ', 'Wall ']],
      dtype='<U5')